# Showcasing in-progress GeoPandas - spherely integration

In [1]:
import geodatasets
import geopandas
import shapely
import spherely

In [2]:
land = geopandas.read_file(geodatasets.get_path("naturalearth.land"))

In [3]:
land.head()

,featurecla,scalerank,min_zoom,geometry
0,Land,1,1.0,"POLYGON ((-59.57209 -80.04018, -59.86585 -80.5..."
1,Land,1,1.0,"POLYGON ((-159.20818 -79.49706, -161.1276 -79...."
2,Land,1,0.0,"POLYGON ((-45.15476 -78.04707, -43.92083 -78.4..."
3,Land,1,1.0,"POLYGON ((-121.21151 -73.50099, -119.91885 -73..."
4,Land,1,1.0,"POLYGON ((-125.55957 -73.48135, -124.03188 -73..."


Converting to a _spherical_ geometry dtype:

In [4]:
land_geog = land.astype({"geometry": geopandas.array.GeometryDtype(engine="spherical")})

In [5]:
land_geog.head()

,featurecla,scalerank,min_zoom,geometry
0,Land,1,1.0,"POLYGON ((-59.5721 -80.04018, -60.09111 -79.83..."
1,Land,1,1.0,"POLYGON ((-160.16789 -79.56563, -159.20818 -79..."
2,Land,1,0.0,"POLYGON ((-44.53779 -78.26259, -45.15476 -78.0..."
3,Land,1,1.0,"POLYGON ((-120.56518 -73.57936, -121.21151 -73..."
4,Land,1,1.0,"POLYGON ((-124.79572 -73.67731, -125.55957 -73..."


In [6]:
land_geog.geometry.dtype

geometry[spherical]

Now calculating the area works without warning and gives you the result in m²!

In [7]:
land.geometry.area

/tmp/ipykernel_934873/3342291871.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  land.geometry.area


0        4.202195
1        3.716703
2       20.382879
3        1.541898
4        0.683017
          ...    
122      2.437153
123     13.689594
124     19.395611
125    104.819482
126    677.509565
Length: 127, dtype: float64

In [8]:
land_geog.geometry.area

0      8.588523e+09
1      8.813227e+09
2      4.552045e+10
3      5.340779e+09
4      2.386070e+09
           ...     
122    4.970397e+09
123    2.952446e+10
124    4.287863e+10
125    2.162656e+11
126    2.190031e+12
Length: 127, dtype: float64

In [9]:
land_geog.distance(spherely.create_point(0, 0))

0      9.443957e+06
1      1.109759e+07
2      9.073724e+06
3      1.087813e+07
4      1.100197e+07
           ...     
122    9.260411e+06
123    1.002959e+07
124    9.921514e+06
125    9.608350e+06
126    7.645315e+06
Length: 127, dtype: float64

In [10]:
land_geog.centroid

0       POINT (-62.28401 -80.48564)
1      POINT (-161.46469 -78.93375)
2        POINT (-47.7858 -79.54124)
3      POINT (-120.92793 -73.73881)
4        POINT (-125.897 -73.56808)
                   ...             
122       POINT (48.43643 80.51494)
123       POINT (96.26788 79.92518)
124      POINT (-91.43212 79.65842)
125      POINT (-79.75826 80.07678)
126      POINT (-41.96454 73.14658)
Length: 127, dtype: geometry

Methods that are not yet implemented raise an error (just need to add an informative message saying they need to cast to GeometryDtype to be able to use this method):

In [11]:
land_geog.buffer(10)

NotImplementedError: 

Binary predicates and overlays also already work:

In [12]:
land_geog.intersects(land_geog.geometry[112])

0      False
1      False
2      False
3      False
4      False
       ...  
122    False
123    False
124    False
125    False
126    False
Length: 127, dtype: bool

In [13]:
land_geog.intersection(land_geog.geometry[112])

0      GEOMETRYCOLLECTION EMPTY
1      GEOMETRYCOLLECTION EMPTY
2      GEOMETRYCOLLECTION EMPTY
3      GEOMETRYCOLLECTION EMPTY
4      GEOMETRYCOLLECTION EMPTY
                 ...           
122    GEOMETRYCOLLECTION EMPTY
123    GEOMETRYCOLLECTION EMPTY
124    GEOMETRYCOLLECTION EMPTY
125    GEOMETRYCOLLECTION EMPTY
126    GEOMETRYCOLLECTION EMPTY
Length: 127, dtype: geometry

In [14]:
land_geog.geometry.geom_type.value_counts()

Polygon    127
Name: count, dtype: int64